# Preparation

In [18]:
import pandas as pd
import numpy as np
import warnings
import gc
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, GroupShuffleSplit, cross_val_predict
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

In [3]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl")

In [59]:
df.head()

,ModelID,SMILES,SequencingID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),...,Bit_1022,Bit_1023,Donor,Acceptor,Aromatic,Hydrophobe,LumpedHydrophobe,PosIonizable,NegIonizable,ZnBinder
0,ACH-000001,B(C1=CC2=CC=CC=C2S1)(O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,2.0,3.0,2.0,0.0,0.0,0.0
1,ACH-000001,B(C1=CC=CC=C1)(C2=CC=CC=C2)OCCN,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,1.0,2.0,2.0,2.0,1.0,0.0,0.0
2,ACH-000001,C#CCCCCCCCCCCCCCCCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,2.0,0.0,16.0,0.0,0.0,1.0,1.0
3,ACH-000001,C(C(=O)O)S,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,0.0,1.0,0.0,0.0,1.0,2.0
4,ACH-000001,C(C(C(=O)O)N)SCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,3.0,4.0,0.0,2.0,0.0,1.0,2.0,2.0


# Early fusion

In [57]:
def early_fusion(df, target, model_name):
    if 'Bit_0' not in df.columns:
        fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index).astype('uint8')
        fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
        df = pd.concat([df, fp_df], axis=1)
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), index=df.index).astype('float32')
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)
        
    # training feature set
    X = pd.concat([df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']],
                        df.loc[:, df.columns.str.startswith('Bit_')],
                        df.filter(regex=r'.* \(.*\)')], axis=1)
    y = df[target]
    
    # train-test split 
    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['DRUG_ID'].values))
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups = df.iloc[train_idx]['DRUG_ID'].values 

    # apply PCA to fingerprint columns
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    mfp_cols = [col for col in df.columns if col.startswith('Bit_')]
    
    preprocessor = ColumnTransformer(
            transformers=[
                ('pharmacophore', StandardScaler(), pharmacophores),
                ('mfp', PCA(n_components=10), mfp_cols)
            ],
            remainder='passthrough',
            verbose_feature_names_out=False
        )
    preprocessor.set_output(transform="pandas")
    
    X_train_transformed = preprocessor.fit_transform(X_train)
    X_test_transformed = preprocessor.transform(X_test)

    if model_name == 'RandomForest':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features='sqrt',
            n_jobs=4,
            random_state=42
        )

        print("Starting Cross-Validation on Training Data...")
        cv_results = cross_validate(
            model, X_train_transformed, y_train, 
            groups=groups, 
            cv=GroupKFold(5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        mse_scores = -cv_results['test_neg_mean_squared_error']
        rmse_scores = np.sqrt(mse_scores)
        r2_scores = cv_results['test_r2']

        print(f"--- Multimodal (Random Forest - Early Fusion) Performance ---")
        print(f"R² Score: {np.mean(r2_scores):.4f}")
        print(f"RMSE:     {np.mean(rmse_scores):.4f}")
        print(f"------------------------------------")

        model.fit(X_train_transformed, y_train)

        y_pred = model.predict(X_test_transformed)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        
        # feature importance
        all_features = X.columns
        importances = model.feature_importances_
        gene_cols = X.filter(regex=r'.* \(.*\)').columns
        
        features_df = pd.DataFrame({
            'Feature_Name': all_features,
            'Importance': importances,
            'Type': ['Genetic' if f in gene_cols else 'Molecular structure' for f in all_features]
        })
        
        selected_features = features_df[features_df['Importance'] > 0.0001].copy()
        chem_selected = selected_features[selected_features['Type'] == 'Molecular structure']
        bio_selected = selected_features[selected_features['Type'] == 'Genetic']
        
        print(f"Total number of selected features: {len(selected_features)} von {len(all_features)}")
        print(f" -> Used chemical features:  {len(chem_selected)}")
        print(f" -> Used biological genes:    {len(bio_selected)}")
        
        print(f"\nTop 10 most important features:")
        print(features_df.sort_values(by='Importance', ascending=False).head(10))

    elif model_name == 'ElasticNet':
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=GroupKFold(n_splits=5).split(X_train, y_train, groups=groups),
                max_iter=8000,
                alphas=20,
                tol=1e-3,
                random_state=42,
                n_jobs=1
            ))
        ])

        pipeline.fit(X_train_transformed, y_train)

        train_r2_global = pipeline.score(X_train_transformed, y_train)
        fitted_model = pipeline.named_steps['model']
        best_alpha_idx = np.where(fitted_model.alphas_ == fitted_model.alpha_)[0][0]
        mean_mse_best_alpha = np.mean(fitted_model.mse_path_[best_alpha_idx])
        variance_y_train = np.var(y_train)
        train_cv_r2 = 1 - (mean_mse_best_alpha / variance_y_train)

        y_pred = pipeline.predict(X_test_transformed)
        test_r2 = pipeline.score(X_test_transformed, y_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        print("\n" + "="*40)
        print(f"--- Elastic Net, Early fusion ---")
        print("="*40)
        print(f"Chosen Alpha:           {fitted_model.alpha_:.6f}")
        print(f"Chosen L1-Ratio:        {fitted_model.l1_ratio_:.2f}")
        print("-"*40)
        print(f"CV R² score:     {train_cv_r2:.4f}")
        print(f"Held-Out test R² score:       {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        print("="*40)

        # analyze feature importance
        final_model = pipeline.named_steps['model']
        coefs = final_model.coef_

        # Create a summary table
        feature_names = X_train_transformed.columns
        features_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
        features_df['Abs_Coef'] = features_df['Coefficient'].abs()

        # Filter for features the model didn't set to zero
        selected_features = features_df[features_df['Coefficient'] != 0]

        print(f"\nElastic Net selected {len(selected_features)} features out of {len(feature_names)}.")
        print(f"Top 5 Positive Features (Increase {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
        print(f"\nTop 5 Negative Features (Decrease {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

## Early fusion with Random Forest

In [44]:
early_fusion(df, target='LN_IC50', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (Random Forest - Early Fusion) Performance ---
R² Score: 0.0320
RMSE:     2.7581
------------------------------------
Test R² Score: 0.1025
Test RMSE:     2.4872
Total number of selected features: 980 von 996
 -> Used chemical features:  18
 -> Used biological genes:    962

Top 10 most important features:
   Feature_Name  Importance                 Type
13         pca5    0.066660  Molecular structure
10         pca2    0.065958  Molecular structure
11         pca3    0.060323  Molecular structure
15         pca7    0.058595  Molecular structure
16         pca8    0.049772  Molecular structure
14         pca6    0.048438  Molecular structure
9          pca1    0.046612  Molecular structure
17         pca9    0.046146  Molecular structure
8          pca0    0.044661  Molecular structure
0         Donor    0.040169  Molecular structure


In [45]:
early_fusion(df, target='AUC', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (Random Forest - Early Fusion) Performance ---
R² Score: -0.0345
RMSE:     0.1519
------------------------------------
Test R² Score: 0.0590
Test RMSE:     0.1280
Total number of selected features: 990 von 996
 -> Used chemical features:  18
 -> Used biological genes:    972

Top 10 most important features:
   Feature_Name  Importance                 Type
13         pca5    0.079215  Molecular structure
16         pca8    0.056749  Molecular structure
11         pca3    0.056301  Molecular structure
15         pca7    0.052298  Molecular structure
9          pca1    0.050523  Molecular structure
12         pca4    0.050426  Molecular structure
10         pca2    0.048420  Molecular structure
17         pca9    0.047563  Molecular structure
14         pca6    0.046712  Molecular structure
8          pca0    0.043304  Molecular structure


## Early fusion with Elastic Net

In [58]:
early_fusion(df, target='LN_IC50', model_name='ElasticNet')


--- Elastic Net, Early fusion ---
Chosen Alpha:           0.589133
Chosen L1-Ratio:        0.05
----------------------------------------
CV R² score:     0.0306
Held-Out test R² score:       0.1034
Test RMSE:     2.4860

Elastic Net selected 204 features out of 996.
Top 5 Positive Features (Increase LN_IC50):
       Feature  Coefficient  Abs_Coef
10        pca2     0.352052  0.352052
3   Hydrophobe     0.193793  0.193793
17        pca9     0.070529  0.070529
11        pca3     0.069201  0.069201
14        pca6     0.048114  0.048114

Top 5 Negative Features (Decrease LN_IC50):
             Feature  Coefficient  Abs_Coef
15              pca7    -0.216515  0.216515
6       NegIonizable    -0.188660  0.188660
4   LumpedHydrophobe    -0.132499  0.132499
1           Acceptor    -0.122801  0.122801
12              pca4    -0.090932  0.090932


In [59]:
early_fusion(df, target='AUC', model_name='ElasticNet')


--- Elastic Net, Early fusion ---
Chosen Alpha:           0.189996
Chosen L1-Ratio:        0.05
----------------------------------------
CV R² score:     -0.0648
Held-Out test R² score:       0.0194
Test RMSE:     0.1307

Elastic Net selected 15 features out of 996.
Top 5 Positive Features (Increase AUC):
          Feature  Coefficient  Abs_Coef
10           pca2     0.009131  0.009131
266   TJP1 (7082)     0.002527  0.002527
810   PPIC (5480)     0.002378  0.002378
178   CTTN (2017)     0.001540  0.001540
621  PTPRF (5792)     0.001334  0.001334

Top 5 Negative Features (Decrease AUC):
           Feature  Coefficient  Abs_Coef
15            pca7    -0.007354  0.007354
13            pca5    -0.004664  0.004664
16            pca8    -0.003287  0.003287
12            pca4    -0.002470  0.002470
906  IKZF1 (10320)    -0.002065  0.002065


# Late fusion

In late fusion, you would take the outputs (predictions or confidence scores) from these two separate models and combine them. This combination can be done in several ways:

    Averaging: Simple or weighted averaging of confidence scores.
    Voting: Each model "votes" for a class, and the majority wins.
    Product Rule: Multiplying probabilities (assuming independence).
    Small Model: Using another simple model (like a logistic regression or a small neural network) that takes the individual predictions as input and learns how to combine them.

I think it's gonna fail because my individual models are not performing well...

### Linear regression

In [66]:
def late_fusion_lr(df, target, model_name):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=42)
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
    X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
    y = df[target].values.astype('float32')

    train_idx_drug, test_idx_drug = next(gss.split(df, groups=df['DRUG_ID']))
    train_idx_cl, test_idx_cl = next(gss.split(df, groups=df['ModelID']))
    df_train_drug = df.iloc[train_idx_drug]
    df_train_cl = df.iloc[train_idx_cl]
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    del df_train_drug, df_train_cl


    gc.collect()
    # single-modal models
    if model_name == 'ElasticNet':
        print("Elastic Net - Late Fusion")
        # training data
        target = 'LN_IC50'
        pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
        X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
        X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
        y = df[target].values.astype('float32')

        model_chem = Pipeline([
            ('scaler', StandardScaler()),
            ('selector', VarianceThreshold(threshold=0.01)),
            ('pca', PCA(n_components=50, random_state=42)),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=4,
                precompute=False
                ))
        ])
        model_genomic = Pipeline([
            ('scaler', StandardScaler()),
            ('selector', VarianceThreshold(threshold=0.01)),
            ('pca', PCA(n_components=50, random_state=42)),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=4,
                precompute=False
                ))
        ])

    elif model_name == 'RandomForest':
        print("Random Forest - Late Fusion")
        model_genomic = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
        model_chem = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
        '''cv_genomic = cross_val_predict(rf_genomic, X_genomic, y,groups=df['ModelID'], cv=GroupKFold(n_splits=5))
        cv_chem = cross_val_predict(rf_chem, X_chem, y,groups=df['DRUG_ID'], cv=GroupKFold(n_splits=5))
        
        X_combined = np.column_stack((cv_genomic, cv_chem))
        # use linear regression for final prediction
        lr = LinearRegression()
        lr.fit(X_combined, y)
        # train baseline models
        print("Fitting Genomic and Chemical Models on Full Training Data...")
        rf_genomic.fit(X_genomic, y)
        rf_chem.fit(X_chem, y)

        # testing
        # using a combined split for final evaluation to ensure no data leakage
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(X_combined, y, groups=df['DRUG_ID']))
        X_test_gene = X_genomic[test_idx]
        X_test_chem = X_chem[test_idx]
        y_test = y[test_idx]
        print("Evaluating on Test Set...")
        y_pred_gen = rf_genomic.predict(X_test_gene)
        y_pred_chem = rf_chem.predict(X_test_chem)
        y_pred_meta = lr.predict(np.column_stack([y_pred_gen, y_pred_chem]))
        
        print("="*40)
        #print(f"R² Genomic single-modality model: {r2_score(y_test, y_pred_gen):.4f}")
        #print(f"R² Chem single-modality model:    {r2_score(y_test, y_pred_chem):.4f}")
        print(f"R² Late Fusion (meta model): {r2_score(y_test, y_pred_meta):.4f}")
        print("="*40)
        
         # Extract Decision-Learner Weights
        weight_chem, weight_gene = lr.coef_
        print("Decision-Learner Weights:")
        print(f" -> Weight for Chemical Model: {weight_chem:.4f}")
        print(f" -> Weight for Genomic Model:  {weight_gene:.4f}")'''
    # OOF predictions
    cv_genomic = cross_val_predict(model_genomic, X_genomic, y, groups=df['ModelID'], cv=GroupKFold(5))
    cv_chem = cross_val_predict(model_chem, X_chem, y, groups=df['DRUG_ID'], cv=GroupKFold(5))

    # split for a clean meta-model evaluation
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['ModelID']))
    X_combined_train = np.column_stack([cv_genomic[train_idx], cv_chem[train_idx]])

    lr = LinearRegression()
    lr.fit(X_combined_train, y[train_idx])
    y_pred_meta = lr.predict(np.column_stack([cv_genomic[test_idx], cv_chem[test_idx]]))
    #y_pred_gen = model_genomic.predict(X_test_gene)
    #y_pred_chem = model_chem.predict(X_test_chem)

    print("="*40)
    #print(f"R² Genomic single-modality model: {r2_score(y[test_idx], y_pred_gen):.4f}")
    #print(f"R² Chem single-modality model:    {r2_score(y[test_idx], y_pred_chem):.4f}")
    print(f"Test R² score: {r2_score(y[test_idx], y_pred_meta):.4f}")
    print(f"Test RMSE: {np.sqrt(mean_squared_error(y[test_idx], y_pred_meta)):.4f}")
    print("="*40)
    
        # Extract Decision-Learner Weights
    weight_chem, weight_gene = lr.coef_
    print("Decision-Learner Weights:")
    print(f" -> Weight for Chemical Model: {weight_chem:.4f}")
    print(f" -> Weight for Genomic Model:  {weight_gene:.4f}")

In [67]:
late_fusion_lr(df, target='LN_IC50', model_name='RandomForest')

Random Forest - Late Fusion
Test R² score: 0.0577
Test RMSE: 2.7276
Decision-Learner Weights:
 -> Weight for Chemical Model: 1.1497
 -> Weight for Genomic Model:  0.4304


In [ ]:
late_fusion_lr(df, target='LN_IC50', model_name='ElasticNet')

Elastic Net - Late Fusion
R² Late Fusion (meta model): 0.0523
RMSE Late Fusion (meta model): 2.7354
Decision-Learner Weights:
 -> Weight for Chemical Model: 1.0600
 -> Weight for Genomic Model:  0.1142


### Simple averaging

In [29]:
def late_fusion_average(df, model, target):
    # training data
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    mfp_cols = [col for col in df.columns if col.startswith('Bit_')]
    X_genomic = df.filter(regex=r'.* \(.*\)')#.values.astype('float32')
    X_chem = df[pharmacophores + mfp_cols]#.values.astype('float32')
    y = df[target]#.values.astype('float32')

    if model == 'ElasticNet':
        # scale only pharmacophore features
        preprocessor = ColumnTransformer(
            transformers=[
                ('pharmacophore', StandardScaler(), pharmacophores),
                ('mfp', 'passthrough', mfp_cols)
            ],
            remainder='drop'
        )
        model_chem = Pipeline([
            ('scaler', preprocessor),
            ('selector', VarianceThreshold(threshold=0.01)),
            ('pca', PCA(n_components=50, random_state=42)),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=4,
                precompute=False
                ))
        ])
        model_genomic = Pipeline([
            ('selector', VarianceThreshold(threshold=0.01)),
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=50, random_state=42)),
            ('model', ElasticNetCV(
                l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=4,
                precompute=False
                ))
        ])
    elif model == 'RandomForest':
        model_genomic = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
        model_chem = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
    # OOF predictions
    cv_genomic = cross_val_predict(model_genomic, X_genomic, y, groups=df['ModelID'], cv=GroupKFold(5))
    cv_chem = cross_val_predict(model_chem, X_chem, y, groups=df['DRUG_ID'], cv=GroupKFold(5))
    # testing
    y_pred_meta_avg_oof = (cv_genomic + cv_chem) / 2

    print("--- Simple Averaging (Honest OOF Baseline) ---")
    print(f"R² Genomic (Strict Cell-Line Split): {r2_score(y, cv_genomic):.4f}")
    print(f"R² Chem (Strict Drug Split):         {r2_score(y, cv_chem):.4f}")
    print(f"R² Average (Combined):               {r2_score(y, y_pred_meta_avg_oof):.4f}")
    print(f"RMSE Average:                        {np.sqrt(mean_squared_error(y, y_pred_meta_avg_oof)):.4f}")

#### Elastic Net

In [ ]:
# training data
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
mfp_cols = [col for col in df.columns if col.startswith('Bit_')]
X_genomic = df.filter(regex=r'.* \(.*\)')#.values.astype('float32')
X_chem = df[pharmacophores + mfp_cols]#.values.astype('float32')
y = df[target]#.values.astype('float32')

# apply PCA to fingerprint columns
bit_pipe = Pipeline([
    ('selector', VarianceThreshold(threshold=0.01)),
    ('pca', PCA(n_components=50, random_state=42))
])
# scale only pharmacophore features
preprocessor = ColumnTransformer(
    transformers=[
        ('pharmacophore', StandardScaler(), pharmacophores),
        ('mfp', 'passthrough', mfp_cols)
    ],
    remainder='drop'
)

pipeline_chem = Pipeline([
    ('scaler', preprocessor),
    ('model', ElasticNetCV(
        l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
        cv=5,
        random_state=42,
        max_iter=7000,
        alphas=20,
        tol=1e-3,
        n_jobs=4,
        precompute=False
        ))
])
pipeline_genomic = Pipeline([
    ('selector', VarianceThreshold(threshold=0.01)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=50, random_state=42)),
    ('model', ElasticNetCV(
        l1_ratio=[0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
        cv=5,
        random_state=42,
        max_iter=7000,
        alphas=20,
        tol=1e-3,
        n_jobs=4,
        precompute=False
        ))
])
# OOF predictions
cv_genomic = cross_val_predict(pipeline_genomic, X_genomic, y, groups=df['ModelID'], cv=GroupKFold(5))
cv_chem = cross_val_predict(pipeline_chem, X_chem, y, groups=df['DRUG_ID'], cv=GroupKFold(5))

Results after simple averaging - and using PCA for fingerprints

In [19]:
y_pred_meta_avg_oof = (cv_genomic + cv_chem) / 2

print("--- Simple Averaging (Honest OOF Baseline) ---")
print(f"R² Genomic (Cell-Line Split): {r2_score(y, cv_genomic):.4f}")
print(f"R² Chem (Drug Split):         {r2_score(y, cv_chem):.4f}")
print(f"R² Average (Combined):               {r2_score(y, y_pred_meta_avg_oof):.4f}")
print(f"RMSE Average:                        {np.sqrt(mean_squared_error(y, y_pred_meta_avg_oof)):.4f}")

--- Simple Averaging (Honest OOF Baseline) ---
R² Genomic (Strict Cell-Line Split): 0.0538
R² Chem (Strict Drug Split):         -0.0512
R² Average (Combined):               0.0685
RMSE Average:                        2.7255


PCA over everything

In [30]:
late_fusion_average(df, model='ElasticNet', target='LN_IC50')

--- Simple Averaging (Honest OOF Baseline) ---
R² Genomic (Strict Cell-Line Split): 0.0538
R² Chem (Strict Drug Split):         0.0183
R² Average (Combined):               0.0805
RMSE Average:                        2.7079


In [31]:
late_fusion_average(df, model='ElasticNet', target='AUC')

--- Simple Averaging (Honest OOF Baseline) ---
R² Genomic (Strict Cell-Line Split): 0.0237
R² Chem (Strict Drug Split):         -0.0408
R² Average (Combined):               0.0192
RMSE Average:                        0.1472


#### Random Forest

In [22]:
late_fusion_average(df, model='RandomForest', target='LN_IC50')

Starting Cross-Validation for Genomic Model...
--- Simple Averaging (Honest OOF Baseline) ---
R² Genomic (Strict Cell-Line Split): 0.0511
R² Chem (Strict Drug Split):         -0.0048
R² Average (Combined):               0.0548
RMSE Average:                        2.7454


In [25]:
late_fusion_average(df, model='RandomForest', target='AUC')

Starting Cross-Validation for Genomic Model...
--- Simple Averaging (Honest OOF Baseline) ---
R² Genomic (Strict Cell-Line Split): 0.0231
R² Chem (Strict Drug Split):         -0.0180
R² Average (Combined):               0.0246
RMSE Average:                        0.1468
